In [ ]:
# | default_exp preprocessing.ocr

In [ ]:
%load_ext autoreload
%autoreload 2

# Document and image OCR preprocessing

> Recursively recognize PDF pages and PNG/JPEG images with local Ollama or Alibaba Cloud DashScope.

Each `.pdf`, `.png`, `.jpg`, or `.jpeg` source produces one UTF-8 Markdown document under a `.md` directory at the source root. Relative directories are preserved, PDF page comments retain page provenance, and files are processed sequentially. Ollama remains the default provider; DashScope uses its OpenAI-compatible multimodal interface when selected.

In [ ]:
# | export
import base64
import os
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from tempfile import NamedTemporaryFile
from time import perf_counter
from typing import Literal, cast

import pymupdf
from dotenv import load_dotenv
from ollama import Client as OllamaClient
from openai import OpenAI
from tqdm.auto import tqdm


In [ ]:
# | export
def _find_project_root() -> Path:
    """Find the nearest parent containing pyproject.toml."""
    starts: list[Path] = []
    module_file = globals().get("__file__")
    if isinstance(module_file, str):
        starts.append(Path(module_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / "pyproject.toml").is_file():
                return candidate
    return Path.cwd().resolve()


PROJ_ROOT = _find_project_root()
load_dotenv(PROJ_ROOT / ".env", override=False)

In [ ]:
# | export
OCRProvider = Literal["ollama", "dashscope"]
_OCRClient = OllamaClient | OpenAI


@dataclass(frozen=True)
class OCRResult:
    """Outcome of attempting to convert one source file to Markdown."""

    pdf_path: Path
    markdown_path: Path
    status: Literal["processed", "skipped", "failed"]
    pages_total: int = 0
    pages_completed: int = 0
    error: str | None = None

    @property
    def source_path(self) -> Path:
        """Source PDF or image path (preferred provider-neutral name)."""
        return self.pdf_path

In [ ]:
# | export
def _resolve_root(root_folder: Path | str) -> Path:
    root = Path(root_folder).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"OCR root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"OCR root is not a directory: {root}")
    return root


_SUPPORTED_SOURCE_SUFFIXES = frozenset({".pdf", ".png", ".jpg", ".jpeg"})
_IMAGE_SUFFIXES = frozenset({".png", ".jpg", ".jpeg"})


def _ocr_jobs(root: Path) -> list[tuple[Path, Path]]:
    """Return deterministic source/target pairs and reject target collisions."""
    output_root = root / ".md"
    source_files = [
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.casefold() in _SUPPORTED_SOURCE_SUFFIXES
        and not path.is_relative_to(output_root)
    ]
    source_files.sort(
        key=lambda path: (
            path.relative_to(root).as_posix().casefold(),
            path.relative_to(root).as_posix(),
        )
    )

    jobs: list[tuple[Path, Path]] = []
    targets: dict[str, Path] = {}
    for source_path in source_files:
        relative_path = source_path.relative_to(root).with_suffix(".md")
        markdown_path = output_root / relative_path
        collision_key = markdown_path.as_posix().casefold()
        if previous := targets.get(collision_key):
            raise ValueError(
                f"OCR output collision: {previous} and {source_path} both map to {markdown_path}"
            )
        targets[collision_key] = source_path
        jobs.append((source_path, markdown_path))
    return jobs

In [ ]:
# | export
_DEFAULT_OLLAMA_MODEL = "glm-ocr"
_DEFAULT_DASHSCOPE_MODEL = "qwen3.7-plus"
_DEFAULT_DASHSCOPE_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
_OLLAMA_PROMPT = "Text Recognition:"
_DASHSCOPE_PROMPT = (
    "Convert this document page to Markdown. Preserve the reading order, headings, "
    "paragraphs, lists, tables, code, and formulas. Return only the Markdown "
    "transcription without commentary."
)
_DASHSCOPE_MAX_DATA_URL_BYTES = 10 * 1024 * 1024


def _resolve_provider(provider: OCRProvider | str) -> OCRProvider:
    normalized = provider.strip().casefold()
    if normalized not in {"ollama", "dashscope"}:
        raise ValueError("provider must be 'ollama' or 'dashscope'")
    return cast(OCRProvider, normalized)


def _resolve_model(provider: OCRProvider, model: str | None) -> str:
    if model is not None:
        if not model.strip():
            raise ValueError("model must not be empty")
        return model.strip()
    if provider == "dashscope":
        return (
            os.getenv("OPENAILIKED_OCR_MODEL", "").strip()
            or _DEFAULT_DASHSCOPE_MODEL
        )
    return _DEFAULT_OLLAMA_MODEL


def _resolve_prompt(provider: OCRProvider, prompt: str | None) -> str:
    if prompt is not None:
        if not prompt.strip():
            raise ValueError("prompt must not be empty")
        return prompt.strip()
    return _DASHSCOPE_PROMPT if provider == "dashscope" else _OLLAMA_PROMPT


def _response_content(response: object, provider: OCRProvider) -> str:
    if provider == "ollama":
        message = getattr(response, "message", None)
    else:
        choices = getattr(response, "choices", None)
        message = getattr(choices[0], "message", None) if choices else None
    content = getattr(message, "content", None)
    if not isinstance(content, str) or not content.strip():
        raise ValueError(f"{provider} OCR returned an empty response")
    return content.strip()


def _data_url(image_bytes: bytes, mime_type: str) -> str:
    encoded = base64.b64encode(image_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def _dashscope_image_data_url(pixmap: pymupdf.Pixmap) -> str:
    png_data_url = _data_url(pixmap.tobytes("png"), "image/png")
    if len(png_data_url.encode("ascii")) <= _DASHSCOPE_MAX_DATA_URL_BYTES:
        return png_data_url

    jpeg_data_url = _data_url(
        pixmap.tobytes("jpeg", jpg_quality=90),
        "image/jpeg",
    )
    if len(jpeg_data_url.encode("ascii")) <= _DASHSCOPE_MAX_DATA_URL_BYTES:
        return jpeg_data_url
    raise ValueError(
        "Image payload exceeds DashScope's 10 MiB Base64 input limit; "
        "reduce PDF dpi or the source image dimensions"
    )


def _pixmap_markdown(
    pixmap: pymupdf.Pixmap,
    provenance: str,
    *,
    client: _OCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
) -> str:
    if provider == "ollama":
        response = client.chat(  # type: ignore[operator]
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                    "images": [pixmap.tobytes("png")],
                }
            ],
            options={"temperature": 0},
        )
    else:
        response = client.chat.completions.create(  # type: ignore[union-attr]
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": _dashscope_image_data_url(pixmap)
                            },
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            temperature=0,
            extra_body={"enable_thinking": False},
        )
    content = _response_content(response, provider)
    return f"<!-- {provenance} -->\n\n{content}"


def _page_markdown(
    page: pymupdf.Page,
    page_number: int,
    *,
    client: _OCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    dpi: int,
) -> str:
    return _pixmap_markdown(
        page.get_pixmap(dpi=dpi, alpha=False),
        f"Page {page_number}",
        client=client,
        provider=provider,
        model=model,
        prompt=prompt,
    )


def _load_image_pixmap(path: Path) -> pymupdf.Pixmap:
    pixmap = pymupdf.Pixmap(str(path))
    if pixmap.colorspace not in {pymupdf.csGRAY, pymupdf.csRGB}:
        pixmap = pymupdf.Pixmap(pymupdf.csRGB, pixmap)
    if pixmap.alpha:
        pixmap = pymupdf.Pixmap(pixmap, 0)
    return pixmap


def _atomic_write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path: Path | None = None
    try:
        with NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="\n",
            dir=path.parent,
            prefix=f".{path.name}.",
            suffix=".tmp",
            delete=False,
        ) as temporary_file:
            temporary_file.write(content)
            temporary_path = Path(temporary_file.name)
        temporary_path.replace(path)
        temporary_path = None
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

In [ ]:
# | export
def ocr_pdf(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _OCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    dpi: int = 200,
    overwrite: bool = False,
) -> OCRResult:
    """Convert one PDF to Markdown with page-level OCR requests.

    The final Markdown path is replaced only after every page succeeds. Exceptions
    are captured in the returned result so a folder batch can continue.
    """
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_prompt = _resolve_prompt(selected_provider, prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")

    if target.exists():
        if not target.is_file():
            return OCRResult(source, target, "failed", error="Markdown target is not a file")
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")

            page_sections: list[str] = []
            for page_number, page in enumerate(document, start=1):
                page_sections.append(
                    _page_markdown(
                        page,
                        page_number,
                        client=client,
                        provider=selected_provider,
                        model=selected_model,
                        prompt=selected_prompt,
                        dpi=dpi,
                    )
                )
                pages_completed = page_number

        markdown = "\n\n".join(page_sections).rstrip() + "\n"
        _atomic_write_text(target, markdown)
        return OCRResult(source, target, "processed", pages_total, pages_completed)
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
def ocr_image(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _OCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    overwrite: bool = False,
) -> OCRResult:
    """Convert one PNG or JPEG image to Markdown with one OCR request."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_prompt = _resolve_prompt(selected_provider, prompt)

    if target.exists():
        if not target.is_file():
            return OCRResult(source, target, "failed", error="Markdown target is not a file")
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    try:
        if not source.is_file():
            raise FileNotFoundError(f"Image does not exist: {source}")
        if source.suffix.casefold() not in _IMAGE_SUFFIXES:
            raise ValueError(f"Unsupported image type: {source.suffix or '<none>'}")

        pixmap = _load_image_pixmap(source)
        pages_total = 1
        markdown = _pixmap_markdown(
            pixmap,
            "Image",
            client=client,
            provider=selected_provider,
            model=selected_model,
            prompt=selected_prompt,
        ).rstrip() + "\n"
        pages_completed = 1
        _atomic_write_text(target, markdown)
        return OCRResult(source, target, "processed", pages_total, pages_completed)
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
def _create_ocr_client(
    provider: OCRProvider,
    *,
    host: str,
    request_timeout_s: float,
) -> _OCRClient:
    if provider == "ollama":
        if not host.strip():
            raise ValueError("host must not be empty")
        return OllamaClient(host=host, timeout=request_timeout_s)

    api_key = os.getenv("DASHSCOPE_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(
            f"DASHSCOPE_API_KEY is not configured in {PROJ_ROOT / '.env'}"
        )
    base_url = (
        os.getenv("DASHSCOPE_API_URL", "").strip()
        or _DEFAULT_DASHSCOPE_BASE_URL
    )
    if not base_url.startswith(("http://", "https://")) or "/compatible-mode/" not in base_url:
        raise RuntimeError(
            "DASHSCOPE_API_URL must be an OpenAI-compatible HTTP(S) endpoint"
        )
    return OpenAI(api_key=api_key, base_url=base_url, timeout=request_timeout_s)


def ocr_folder(
    root_folder: Path | str,
    *,
    provider: OCRProvider = "ollama",
    host: str = "http://127.0.0.1:11434",
    model: str | None = None,
    prompt: str | None = None,
    dpi: int = 200,
    overwrite: bool = False,
    request_timeout_s: float = 180,
    client: _OCRClient | None = None,
) -> list[OCRResult]:
    """Recursively OCR PDFs and PNG/JPEG images under ``root_folder``."""
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_prompt = _resolve_prompt(selected_provider, prompt)
    root = _resolve_root(root_folder)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")

    jobs = _ocr_jobs(root)
    if not jobs:
        print(f"No PDF, PNG, or JPEG files found under {root}")
        return []

    ocr_client = (
        client
        if client is not None
        else _create_ocr_client(
            selected_provider,
            host=host,
            request_timeout_s=request_timeout_s,
        )
    )
    if selected_provider == "ollama":
        try:
            ocr_client.show(selected_model)  # type: ignore[union-attr]
        except Exception as error:
            raise RuntimeError(
                f"Cannot use Ollama model '{selected_model}' at {host}. "
                f"Ensure Ollama is running and run `ollama pull {selected_model}`. "
                f"Original error: {error}"
            ) from error

    results: list[OCRResult] = []
    description = f"OCR files ({selected_provider})"
    for source_path, markdown_path in tqdm(jobs, desc=description, unit="file"):
        started_at = perf_counter()
        common_arguments = {
            "client": ocr_client,
            "provider": selected_provider,
            "model": selected_model,
            "prompt": selected_prompt,
            "overwrite": overwrite,
        }
        if source_path.suffix.casefold() == ".pdf":
            result = ocr_pdf(
                source_path,
                markdown_path,
                dpi=dpi,
                **common_arguments,
            )
        else:
            result = ocr_image(
                source_path,
                markdown_path,
                **common_arguments,
            )
        elapsed_s = perf_counter() - started_at
        results.append(result)
        tqdm.write(
            f"OCR task: {source_path} | status={result.status} | "
            f"elapsed={elapsed_s:.2f}s"
        )

    counts = Counter(result.status for result in results)
    print(
        f"OCR complete: {counts['processed']} processed, "
        f"{counts['skipped']} skipped, {counts['failed']} failed"
    )
    return results

## Configuration and batch execution

Set `OCR_PROVIDER` to `"ollama"` or `"dashscope"`, set `PDF_ROOT` to the root containing PDFs and/or images, then uncomment the final line. Supported images are PNG, JPG, and JPEG, including uppercase extensions. Ollama requires `ollama pull glm-ocr`. DashScope reads `DASHSCOPE_API_KEY`, `DASHSCOPE_API_URL`, and optional `OPENAILIKED_OCR_MODEL` from `PROJ_ROOT/.env`; its model defaults to `qwen3.7-plus`. Because both providers share the same `.md` output tree, set `OVERWRITE = True` to regenerate an existing result with another provider.

In [ ]:
PDF_ROOT = Path("../res/PDF-20260721")  # Root containing PDFs and/or images.
OCR_PROVIDER: OCRProvider = "ollama"  # Change to "dashscope" for Aliyun.
OLLAMA_HOST = "http://127.0.0.1:11434"
OCR_MODEL: str | None = None  # Use the provider or environment default.
OCR_DPI = 200
REQUEST_TIMEOUT_S = 180
OVERWRITE = False

In [ ]:
# | notest
results = ocr_folder(
    PDF_ROOT,
    provider=OCR_PROVIDER,
    host=OLLAMA_HOST,
    model=OCR_MODEL,
    dpi=OCR_DPI,
    overwrite=OVERWRITE,
    request_timeout_s=REQUEST_TIMEOUT_S,
)


In [ ]:
# | notest
results

## Tests

The tests use temporary PDFs/images and fake Ollama/OpenAI-compatible clients, so they do not require a running service, incur cloud charges, or write to the repository.

In [ ]:
# | hide
from contextlib import redirect_stdout
from io import StringIO
from types import SimpleNamespace
from tempfile import TemporaryDirectory
from unittest.mock import patch

from fastcore.test import test_eq, test_fail


class FakeOllamaClient:
    def __init__(self, responses=(), show_error: Exception | None = None):
        self.responses = list(responses)
        self.show_error = show_error
        self.show_calls = []
        self.chat_calls = []

    def show(self, model):
        self.show_calls.append(model)
        if self.show_error is not None:
            raise self.show_error
        return {}

    def chat(self, **kwargs):
        self.chat_calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat call")
        response = self.responses.pop(0)
        if isinstance(response, Exception):
            raise response
        return SimpleNamespace(message=SimpleNamespace(content=response))


class FakeOpenAIClient:
    def __init__(self, responses=()):
        self.responses = list(responses)
        self.calls = []
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create)
        )

    def _create(self, **kwargs):
        self.calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat completion call")
        response = self.responses.pop(0)
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=response))]
        )


def make_pdf(path: Path, labels=("page",), password: str | None = None):
    document = pymupdf.open()
    for label in labels:
        page = document.new_page()
        page.insert_text((72, 72), label)
    if password is None:
        document.save(path)
    else:
        document.save(
            path,
            encryption=pymupdf.PDF_ENCRYPT_AES_256,
            owner_pw="owner-password",
            user_pw=password,
        )
    document.close()


def make_image(path: Path, label: str = "image"):
    document = pymupdf.open()
    page = document.new_page(width=240, height=120)
    page.insert_text((24, 60), label)
    page.get_pixmap(alpha=False).save(path)
    document.close()

In [ ]:
# | hide
def test_source_discovery_and_mapping():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        (root / "nested").mkdir()
        (root / ".md").mkdir()
        make_pdf(root / "B.PDF")
        make_pdf(root / "nested" / "a.pdf")
        make_image(root / "nested" / "C.PNG")
        make_image(root / "photo.JpEg")
        make_pdf(root / ".md" / "ignored.pdf")
        make_image(root / ".md" / "ignored.jpg")
        (root / "notes.txt").write_text("not a PDF", encoding="utf-8")

        jobs = _ocr_jobs(root)
        test_eq(
            [source.relative_to(root).as_posix() for source, _ in jobs],
            ["B.PDF", "nested/a.pdf", "nested/C.PNG", "photo.JpEg"],
        )
        test_eq(
            [target.relative_to(root).as_posix() for _, target in jobs],
            [".md/B.md", ".md/nested/a.md", ".md/nested/C.md", ".md/photo.md"],
        )


def test_output_collision_is_rejected():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        make_pdf(root / "same.pdf")
        make_image(root / "same.PNG")
        test_fail(lambda: _ocr_jobs(root), contains="output collision")


test_source_discovery_and_mapping()
test_output_collision_is_rejected()

In [ ]:
# | hide
def test_ocr_pdf_writes_ordered_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "two-pages.pdf"
        markdown_path = root / ".md" / "two-pages.md"
        make_pdf(pdf_path, ("first", "second"))
        client = FakeOllamaClient(("# First", "Second"))

        result = ocr_pdf(pdf_path, markdown_path, client=client)

        test_eq(result.status, "processed")
        test_eq(result.pages_total, 2)
        test_eq(result.pages_completed, 2)
        test_eq(
            markdown_path.read_text(encoding="utf-8"),
            "<!-- Page 1 -->\n\n# First\n\n<!-- Page 2 -->\n\nSecond\n",
        )
        test_eq(len(client.chat_calls), 2)
        for call in client.chat_calls:
            test_eq(call["model"], "glm-ocr")
            test_eq(call["options"], {"temperature": 0})
            test_eq(call["messages"][0]["content"], "Text Recognition:")
            assert isinstance(call["messages"][0]["images"][0], bytes)


def test_skip_and_overwrite():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "one.pdf"
        markdown_path = root / "one.md"
        make_pdf(pdf_path)
        markdown_path.write_text("existing", encoding="utf-8")

        skipped_client = FakeOllamaClient()
        skipped = ocr_pdf(pdf_path, markdown_path, client=skipped_client)
        test_eq(skipped.status, "skipped")
        test_eq(skipped_client.chat_calls, [])
        test_eq(markdown_path.read_text(encoding="utf-8"), "existing")

        overwritten = ocr_pdf(
            pdf_path,
            markdown_path,
            client=FakeOllamaClient(("replacement",)),
            overwrite=True,
        )
        test_eq(overwritten.status, "processed")
        assert "replacement" in markdown_path.read_text(encoding="utf-8")


test_ocr_pdf_writes_ordered_markdown()
test_skip_and_overwrite()

In [ ]:
# | hide
def test_ocr_image_writes_markdown_for_both_providers():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        png_path = root / "scan.png"
        png_markdown = root / "scan.md"
        make_image(png_path, "local image")
        ollama_client = FakeOllamaClient(("# Local image",))

        local = ocr_image(png_path, png_markdown, client=ollama_client)

        test_eq(local.status, "processed")
        test_eq(local.source_path, png_path.resolve())
        test_eq((local.pages_total, local.pages_completed), (1, 1))
        test_eq(
            png_markdown.read_text(encoding="utf-8"),
            "<!-- Image -->\n\n# Local image\n",
        )
        image_bytes = ollama_client.chat_calls[0]["messages"][0]["images"][0]
        assert image_bytes.startswith(b"\x89PNG\r\n\x1a\n")

        jpg_path = root / "photo.jpg"
        jpg_markdown = root / "photo.md"
        make_image(jpg_path, "cloud image")
        dashscope_client = FakeOpenAIClient(("Cloud image",))
        cloud = ocr_image(
            jpg_path,
            jpg_markdown,
            client=dashscope_client,
            provider="dashscope",
        )

        test_eq(cloud.status, "processed")
        data_url = dashscope_client.calls[0]["messages"][0]["content"][0]["image_url"]["url"]
        assert data_url.startswith("data:image/png;base64,")
        assert base64.b64decode(data_url.split(",", 1)[1]).startswith(b"\x89PNG")


def test_folder_processes_mixed_pdfs_and_images():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "document.pdf")
        make_image(root / "scan.png")
        client = FakeOllamaClient(("PDF text", "Image text"))

        captured_output = StringIO()
        with redirect_stdout(captured_output):
            results = ocr_folder(root, client=client)

        test_eq([result.status for result in results], ["processed", "processed"])
        test_eq([result.source_path.name for result in results], ["document.pdf", "scan.png"])
        assert (root / ".md" / "document.md").is_file()
        assert (root / ".md" / "scan.md").is_file()
        test_eq(client.show_calls, ["glm-ocr"])
        timing_output = captured_output.getvalue()
        for source_name in ("document.pdf", "scan.png"):
            source_path = (root / source_name).resolve()
            assert f"OCR task: {source_path} | status=processed | elapsed=" in timing_output


test_ocr_image_writes_markdown_for_both_providers()
test_folder_processes_mixed_pdfs_and_images()

In [ ]:
# | hide
def test_provider_and_model_resolution():
    assert (PROJ_ROOT / "pyproject.toml").is_file()
    test_eq(_resolve_provider("OLLAMA"), "ollama")
    test_fail(lambda: _resolve_provider("unknown"), contains="provider must")
    test_eq(_resolve_model("ollama", None), "glm-ocr")
    test_fail(lambda: _resolve_model("ollama", "  "), contains="model must")
    test_eq(_resolve_prompt("dashscope", " custom "), "custom")

    with patch.dict(os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False):
        test_eq(_resolve_model("dashscope", None), "qwen3.7-plus")
    with patch.dict(
        os.environ,
        {"OPENAILIKED_OCR_MODEL": "environment-model"},
        clear=False,
    ):
        test_eq(_resolve_model("dashscope", None), "environment-model")
        test_eq(_resolve_model("dashscope", "explicit-model"), "explicit-model")


test_provider_and_model_resolution()

In [ ]:
# | hide
def test_dashscope_ocr_payload_and_response():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "cloud.pdf"
        markdown_path = root / "cloud.md"
        make_pdf(pdf_path, ("cloud OCR",))
        client = FakeOpenAIClient(("# Cloud OCR",))
        with patch.dict(
            os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False
        ):
            result = ocr_pdf(
                pdf_path,
                markdown_path,
                client=client,
                provider="dashscope",
            )

        test_eq(result.status, "processed")
        test_eq(len(client.calls), 1)
        call = client.calls[0]
        test_eq(call["model"], "qwen3.7-plus")
        test_eq(call["temperature"], 0)
        test_eq(call["extra_body"], {"enable_thinking": False})
        content = call["messages"][0]["content"]
        image_url = content[0]["image_url"]["url"]
        assert image_url.startswith("data:image/png;base64,")
        assert base64.b64decode(image_url.split(",", 1)[1]).startswith(
            b"\x89PNG\r\n\x1a\n"
        )
        assert "Markdown" in content[1]["text"]
        assert "# Cloud OCR" in markdown_path.read_text(encoding="utf-8")


test_dashscope_ocr_payload_and_response()

In [ ]:
# | hide
class CompressiblePixmap:
    def tobytes(self, output="png", jpg_quality=95):
        return b"x" * (32 if output == "png" else 4)


class OversizedPixmap:
    def tobytes(self, output="png", jpg_quality=95):
        return b"x" * 32


def test_dashscope_image_size_fallback_and_failure():
    with patch.dict(globals(), {"_DASHSCOPE_MAX_DATA_URL_BYTES": 60}):
        data_url = _dashscope_image_data_url(CompressiblePixmap())
        assert data_url.startswith("data:image/jpeg;base64,")
        test_fail(
            lambda: _dashscope_image_data_url(OversizedPixmap()),
            contains="10 MiB",
        )


test_dashscope_image_size_fallback_and_failure()

In [ ]:
# | hide
def test_failures_do_not_publish_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "empty-response.pdf"
        markdown_path = root / "empty-response.md"
        make_pdf(pdf_path)
        result = ocr_pdf(pdf_path, markdown_path, client=FakeOllamaClient(("  ",)))
        test_eq(result.status, "failed")
        assert "empty response" in (result.error or "")
        assert not markdown_path.exists()
        assert not list(root.glob("*.tmp"))


def test_corrupt_image_fails_without_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "corrupt.png"
        markdown_path = root / "corrupt.md"
        image_path.write_bytes(b"not an image")

        result = ocr_image(image_path, markdown_path, client=FakeOllamaClient())

        test_eq(result.status, "failed")
        assert not markdown_path.exists()


def test_corrupt_and_encrypted_pdfs_fail_cleanly():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        corrupt_pdf = root / "corrupt.pdf"
        corrupt_pdf.write_bytes(b"not a PDF")
        encrypted_pdf = root / "encrypted.pdf"
        make_pdf(encrypted_pdf, password="secret")

        corrupt = ocr_pdf(corrupt_pdf, root / "corrupt.md", client=FakeOllamaClient())
        encrypted = ocr_pdf(encrypted_pdf, root / "encrypted.md", client=FakeOllamaClient())
        test_eq(corrupt.status, "failed")
        test_eq(encrypted.status, "failed")
        assert "password" in (encrypted.error or "").lower()
        assert not (root / "corrupt.md").exists()
        assert not (root / "encrypted.md").exists()


def test_folder_continues_after_a_failed_pdf():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "a.pdf")
        make_pdf(root / "b.pdf")
        client = FakeOllamaClient((RuntimeError("model failure"), "# B"))

        results = ocr_folder(root, client=client)

        test_eq([result.status for result in results], ["failed", "processed"])
        assert not (root / ".md" / "a.md").exists()
        assert (root / ".md" / "b.md").exists()
        test_eq(client.show_calls, ["glm-ocr"])


def test_dashscope_failures_are_isolated():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        malformed_pdf = root / "malformed.pdf"
        make_pdf(malformed_pdf)
        malformed_target = root / "malformed.md"
        malformed = ocr_pdf(
            malformed_pdf,
            malformed_target,
            client=FakeOpenAIClient((SimpleNamespace(choices=[]),)),
            provider="dashscope",
        )
        test_eq(malformed.status, "failed")
        assert "empty response" in (malformed.error or "")
        assert not malformed_target.exists()

        make_pdf(root / "a.pdf")
        make_pdf(root / "b.pdf")
        client = FakeOpenAIClient((RuntimeError("cloud failure"), "# B"))
        with patch.dict(
            os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False
        ):
            results = ocr_folder(root, provider="dashscope", client=client)
        result_by_name = {result.pdf_path.name: result for result in results}
        test_eq(result_by_name["a.pdf"].status, "failed")
        test_eq(result_by_name["b.pdf"].status, "processed")
        assert not (root / ".md" / "a.md").exists()
        assert (root / ".md" / "b.md").exists()


test_failures_do_not_publish_markdown()
test_corrupt_image_fails_without_markdown()
test_corrupt_and_encrypted_pdfs_fail_cleanly()
test_folder_continues_after_a_failed_pdf()
test_dashscope_failures_are_isolated()

In [ ]:
# | hide
def test_folder_validation_and_preflight():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        missing = root / "missing"
        test_fail(lambda: ocr_folder(missing), contains="does not exist")

        empty_client = FakeOllamaClient()
        test_eq(ocr_folder(root, client=empty_client), [])
        test_eq(empty_client.show_calls, [])

        make_pdf(root / "document.pdf")
        unavailable_client = FakeOllamaClient(show_error=RuntimeError("offline"))
        test_fail(
            lambda: ocr_folder(root, client=unavailable_client),
            contains="ollama pull glm-ocr",
        )
        test_fail(
            lambda: ocr_folder(root, provider="invalid"),
            contains="provider must",
        )

        with patch.dict(os.environ, {"DASHSCOPE_API_KEY": ""}, clear=False):
            test_fail(
                lambda: ocr_folder(root, provider="dashscope"),
                contains="DASHSCOPE_API_KEY",
            )
        with patch.dict(
            os.environ,
            {
                "DASHSCOPE_API_KEY": "test-key",
                "DASHSCOPE_API_URL": "https://example.invalid/v1",
            },
            clear=False,
        ):
            test_fail(
                lambda: ocr_folder(root, provider="dashscope"),
                contains="OpenAI-compatible",
            )


test_folder_validation_and_preflight()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()